# Creating Custom Networks in SocialJax

This tutorial shows you how to create custom neural network architectures for multi-agent reinforcement learning.

## Overview

SocialJax supports two types of network architectures:

1. **CNN Networks**: For visual observations (grid-based environments)
2. **MLP Networks**: For vector observations (state-based environments)

We'll cover:
- Using the network registry
- Creating custom CNN architectures
- Creating custom MLP architectures
- Registering networks for use with algorithms

## 1. Import Required Components

In [ ]:
# Import JAX and Flax
import jax
import jax.numpy as jnp
import flax.linen as nn
from typing import Sequence, Tuple, Optional

# SocialJax imports
from socialjax.networks import (
    register_network,
    create_network,
    list_networks,
    get_network_class,
)

## 2. Explore Built-in Networks

Let's first see what networks are available.

In [ ]:
# List all registered networks
print("Available networks:")
for name in list_networks():
    print(f"  - {name}")

## 3. Create a Network Using the Factory

In [ ]:
# Create a CNN actor-critic network
network = create_network(
    "cnn_small",  # Network name
    action_dim=8,  # Number of actions
)

print(f"Network type: {type(network).__name__}")

# Initialize and test forward pass
key = jax.random.PRNGKey(0)
dummy_obs = jnp.zeros((15, 15, 3))  # Typical observation shape
params = network.init(key, dummy_obs)

# Forward pass
action_logits, value = network.apply(params, dummy_obs)
print(f"Action logits shape: {action_logits.shape}")
print(f"Value shape: {value.shape}")

## 4. Creating a Custom CNN Network

Let's create a custom CNN architecture with residual connections.

In [ ]:
class ResidualBlock(nn.Module):
    """Residual block with two conv layers."""
    channels: int
    
    @nn.compact
    def __call__(self, x):
        residual = x
        x = nn.Conv(self.channels, kernel_size=(3, 3), padding='SAME')(x)
        x = nn.relu(x)
        x = nn.Conv(self.channels, kernel_size=(3, 3), padding='SAME')(x)
        return nn.relu(x + residual)


@register_network("custom_resnet")
class CustomResNetActorCritic(nn.Module):
    """Custom ResNet-style actor-critic network.
    
    Features:
    - Initial conv layer
    - Multiple residual blocks
    - Separate actor and critic heads
    """
    action_dim: int
    hidden_channels: int = 64
    num_blocks: int = 2
    hidden_size: int = 128
    
    @nn.compact
    def __call__(self, x):
        # Initial convolution
        x = nn.Conv(self.hidden_channels, kernel_size=(7, 7), padding='SAME')(x)
        x = nn.relu(x)
        
        # Residual blocks
        for _ in range(self.num_blocks):
            x = ResidualBlock(self.hidden_channels)(x)
        
        # Pooling
        x = jnp.mean(x, axis=(0, 1))  # Global average pooling
        
        # Actor head
        actor = nn.Dense(self.hidden_size)(x)
        actor = nn.relu(actor)
        action_logits = nn.Dense(self.action_dim)(actor)
        
        # Critic head
        critic = nn.Dense(self.hidden_size)(x)
        critic = nn.relu(critic)
        value = nn.Dense(1)(critic)
        
        return action_logits, jnp.squeeze(value, axis=-1)

## 5. Test the Custom Network

In [ ]:
# Verify registration
print("Updated network list:")
for name in list_networks():
    print(f"  - {name}")

# Create and test
custom_net = create_network("custom_resnet", action_dim=8)

# Initialize
key = jax.random.PRNGKey(0)
dummy_obs = jnp.zeros((15, 15, 3))
params = custom_net.init(key, dummy_obs)

# Forward pass
action_logits, value = custom_net.apply(params, dummy_obs)
print(f"\nCustom ResNet outputs:")
print(f"  Action logits: {action_logits.shape}")
print(f"  Value: {value.shape}")

## 6. Creating a Custom MLP Network

For vector observations, we create MLP networks.

In [ ]:
@register_network("custom_mlp")
class CustomMLPActorCritic(nn.Module):
    """Custom MLP actor-critic network.
    
    Features:
    - Configurable layer sizes
    - Layer normalization for stability
    - Optional dropout
    """
    action_dim: int
    layer_sizes: Tuple[int, ...] = (128, 128)
    use_layer_norm: bool = True
    
    @nn.compact
    def __call__(self, x, training: bool = True):
        # Shared layers
        for size in self.layer_sizes:
            x = nn.Dense(size)(x)
            if self.use_layer_norm:
                x = nn.LayerNorm()(x)
            x = nn.relu(x)
        
        # Actor head
        action_logits = nn.Dense(self.action_dim)(x)
        
        # Critic head
        value = nn.Dense(1)(x)
        
        return action_logits, jnp.squeeze(value, axis=-1)

## 7. Test the Custom MLP Network

In [ ]:
# Create MLP network
mlp_net = create_network(
    "custom_mlp",
    action_dim=4,
    layer_sizes=(64, 64),
)

# Initialize with vector input
key = jax.random.PRNGKey(0)
dummy_state = jnp.zeros(16)  # 16-dimensional state
params = mlp_net.init(key, dummy_state)

# Forward pass
action_logits, value = mlp_net.apply(params, dummy_state)
print(f"Custom MLP outputs:")
print(f"  Action logits: {action_logits.shape}")
print(f"  Value: {value.shape}")

## 8. Network with Attention

Let's create a more advanced network with attention mechanism for multi-agent scenarios.

In [ ]:
class MultiHeadAttention(nn.Module):
    """Simple multi-head attention."""
    num_heads: int = 4
    head_dim: int = 16
    
    @nn.compact
    def __call__(self, x):
        total_dim = self.num_heads * self.head_dim
        qkv = nn.Dense(total_dim * 3)(x)
        q, k, v = jnp.split(qkv, 3, axis=-1)
        
        # Reshape for multi-head attention
        batch_size = x.shape[0] if len(x.shape) > 1 else 1
        q = q.reshape(batch_size, self.num_heads, self.head_dim)
        k = k.reshape(batch_size, self.num_heads, self.head_dim)
        v = v.reshape(batch_size, self.num_heads, self.head_dim)
        
        # Scaled dot-product attention
        scores = jnp.matmul(q, k.transpose(-1, -2)) / jnp.sqrt(self.head_dim)
        attn = jax.nn.softmax(scores, axis=-1)
        out = jnp.matmul(attn, v)
        
        return out.reshape(batch_size, -1)


@register_network("attention_net")
class AttentionActorCritic(nn.Module):
    """Actor-critic with attention mechanism."""
    action_dim: int
    hidden_size: int = 128
    num_heads: int = 4
    
    @nn.compact
    def __call__(self, x):
        # Initial embedding
        x = nn.Dense(self.hidden_size)(x)
        x = nn.relu(x)
        
        # Attention layer
        x = MultiHeadAttention(
            num_heads=self.num_heads,
            head_dim=self.hidden_size // self.num_heads,
        )(x)
        x = nn.relu(x)
        
        # Actor head
        action_logits = nn.Dense(self.action_dim)(x)
        
        # Critic head
        value = nn.Dense(1)(x)
        
        return action_logits, jnp.squeeze(value, axis=-1)

## 9. Using Custom Networks with Algorithms

In [ ]:
# Custom networks can be used in custom algorithms
print("Using custom networks:")
print("""\nfrom socialjax.algorithms import register_algorithm
from socialjax.networks import create_network

@register_algorithm('my_custom_algo')
class MyCustomAlgorithm(BaseAlgorithm):
    def _build_network(self):
        return create_network(
            'custom_resnet',  # Use our custom network
            action_dim=self.action_space.n,
            hidden_channels=32,
            num_blocks=3,
        )
""")

## 10. Network Best Practices

In [ ]:
print("Network Design Tips:")
print("""\n1. Use @nn.compact for concise network definitions
2. Always register networks with @register_network for registry access
3. Use orthogonal initialization for policy networks:
   kernel_init=nn.initializers.orthogonal(scale=0.01)

4. Use layer normalization for stable training

5. For actor-critic networks, return:
   - action_logits: Shape (action_dim,)
   - value: Shape () (scalar)

6. Test with dummy inputs to verify shapes
""")

## Summary

In this tutorial, you learned:

1. **Network Registry**: Using `@register_network` decorator
2. **CNN Networks**: Creating convolutional architectures for visual observations
3. **MLP Networks**: Creating fully-connected architectures for state observations
4. **Advanced Architectures**: Residual blocks, attention mechanisms
5. **Integration**: Using custom networks with algorithms

## Next Steps

- **Tutorial 4**: Using callbacks for logging and monitoring
- **Tutorial 5**: Advanced configuration options